- Measure accuracy for devices
- Home settings

In [1]:
from __future__ import annotations
import pandas as pd 
from typing import Dict, List, Tuple
from pathlib import Path
from collections.abc import Iterable
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.preprocessing import StandardScaler
import copy
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
import json

import gc
import time
import os
import sys
import subprocess


In [2]:
DATASET_FOLDER = "../../../dataset/processed-data"
DEVICE_MAPPING_FOLDER = DATASET_FOLDER + "/device-mapping"
INPUT_FOLDER = DATASET_FOLDER + "/input-data"
DATASET1 = "dataset1"
DATASET2 = "dataset2"
DATASET3 = "dataset3"

DATASET_FOLDER = "../../../dataset/processed-data"
DEVICE_MAPPING_FOLDER = DATASET_FOLDER + "/device-mapping"
DATASET1 = "dataset1"
DATASET2 = "dataset2"
DATASET3 = "dataset3"


DEVICE_TO_BE_INCLUDE_1 = f"{DATASET_FOLDER}/device-mapping/{DATASET1}/device_to_be_include.json"
DEVICE_TO_BE_INCLUDE_2 = f"{DATASET_FOLDER}/device-mapping/{DATASET2}/device_to_be_include.json"
DEVICE_TO_BE_INCLUDE_3 = f"{DATASET_FOLDER}/device-mapping/{DATASET3}/device_to_be_include.json"  

ALL_DATA = f"{DATASET_FOLDER}/input-data/{DATASET3}"


In [3]:
# csv_files_1 = [f"{INPUT_1}/{f.name}" for f in sorted(Path(INPUT_1).iterdir()) if f.is_file() and f.suffix == ".csv"]
# for file in csv_files_1:
#     print(file)

In [4]:
# MAPPING_LIST_1 = f"{DEVICE_MAPPING_FOLDER}/{DATASET1}/list_of_devices_processed.csv"
# device_mapping_1 = pd.read_csv(MAPPING_LIST_1)
# device_dict_1 = (
#     device_mapping_1
#     .set_index("mac_address")
#     .drop(columns=["mac_address"], errors="ignore")
#     .to_dict(orient="index")
# )

In [5]:
device_dict_1 = {}
with open(DEVICE_TO_BE_INCLUDE_3, "r") as f:
    device_dict_1 = json.load(f)

In [6]:
device_dict_1

{'00:01:c0:18:7f:9b': {'device_name': 'Gateway',
  'ip_address': '192.168.0.1',
  'is_iot_device': 0,
  'device_type': 'gateway',
  'vendor': None},
 'f4:f2:6d:ce:9a:5d': {'device_name': 'GoogleOnHub',
  'ip_address': '192.168.0.2',
  'is_iot_device': 1,
  'device_type': 'router',
  'vendor': 'Google'},
 'd0:52:a8:63:47:9e': {'device_name': 'SamsungSmartThingsHub',
  'ip_address': '192.168.0.4',
  'is_iot_device': 1,
  'device_type': 'smart home hub',
  'vendor': 'Samsung'},
 '00:17:88:21:f7:e4': {'device_name': 'PhilipsHUEHub',
  'ip_address': '192.168.0.5',
  'is_iot_device': 1,
  'device_type': 'smart lighting hub',
  'vendor': 'Philips'},
 '54:4a:16:f9:54:18': {'device_name': 'InsteonHub',
  'ip_address': '192.168.0.6',
  'is_iot_device': 1,
  'device_type': 'smart home hub',
  'vendor': 'Insteon'},
 '5c:aa:fd:6c:e0:d4': {'device_name': 'Sonos',
  'ip_address': '192.168.0.7',
  'is_iot_device': 1,
  'device_type': 'smart speaker',
  'vendor': 'Sonos'},
 'e4:71:85:25:ce:ec': {'devic

In [7]:
FEATURE_COLUMNS = [
    "packet_count",
    "total_bytes",
]

DEVICE_COLUMN = "eth.src"          # Replace with "eth.src" if applicable
TIME_COLUMN = "timestamp"
LABEL_COLUMN = "is_iot_device"

KPLITS = 5
RANDOM_STATE = 42
WINDOW_SECONDS = 4

In [8]:
# Outgoing volume
def extract_data_from_pcap(
    file_paths: Iterable[str]
) -> pd.DataFrame:
    columns = [
        "frame.number",
        "frame.time_epoch",
        "frame.len",
        "eth.src",
        "eth.dst",
    ]

    all_data = []

    for file_path in file_paths:
        raw_df = pd.read_csv(
            file_path,
            usecols=columns,
            dtype={
                "frame.number": "int32",
                "frame.time_epoch": "float64",
                "frame.len": "int32",
                "eth.src": "category",
                "eth.dst": "category",
            }
        )

        raw_df["capture_id"] = Path(file_path).stem
        all_data.append(raw_df)


    res = pd.concat(all_data, ignore_index=True)
    res = res.sort_values(
        ["frame.time_epoch", "eth.src"],
        ignore_index=True,
    )

    return res


In [9]:
GATEWAY_MAC = "14:cc:20:51:33:ea"

In [10]:
PARQUET_PATH = ALL_DATA + "/all_data.parquet"

# DuckDB scans the parquet directly and returns only the aggregate.
# It does NOT load the packet-level dataset into pandas.
try:
    import duckdb
except ImportError:
    print("[SETUP] duckdb is not installed. Installing it once...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "duckdb"]
    )
    import duckdb


def build_pairwise_traffic_summary(
    parquet_path: str,
    known_macs,
    memory_limit: str = "2GB",
    threads: int = 4,
) -> pd.DataFrame:
    """
    Aggregate packet-level traffic directly from parquet.

    Only rows where at least one endpoint is a known device are retained.
    The raw packet table is never materialized as a pandas DataFrame.
    """
    known_macs = list(known_macs)
    if not known_macs:
        raise ValueError("known_macs is empty")

    if not Path(parquet_path).exists():
        raise FileNotFoundError(parquet_path)

    parquet_size_gb = Path(parquet_path).stat().st_size / 1024**3

    print(f"[START] Reading directly from: {parquet_path}")
    print(f"[INFO] parquet file size: {parquet_size_gb:.2f} GB")
    print(f"[INFO] known devices: {len(known_macs):,}")
    print(f"[INFO] DuckDB memory limit: {memory_limit}")
    print(f"[INFO] DuckDB threads: {threads}")
    print("[INFO] Raw packet rows will NOT be loaded into pandas.")

    start = time.perf_counter()

    con = duckdb.connect(database=":memory:")
    try:
        # Keep DuckDB itself bounded. If an operation needs more room,
        # DuckDB can spill temporary data to disk.
        con.execute(f"SET memory_limit = '{memory_limit}'")
        con.execute(f"SET threads = {int(threads)}")
        con.execute("SET preserve_insertion_order = false")

        temp_dir = str(Path(".duckdb_tmp").resolve())
        Path(temp_dir).mkdir(parents=True, exist_ok=True)
        con.execute(f"SET temp_directory = '{temp_dir.replace(chr(39), chr(39)*2)}'")

        known_df = pd.DataFrame({"mac": known_macs})
        con.register("known_macs", known_df)

        escaped_path = parquet_path.replace("'", "''")

        print("[SCAN] Aggregating source/destination MAC pairs...")

        pairwise_df = con.execute(
            f"""
            SELECT
                p."eth.src" AS "eth.src",
                p."eth.dst" AS "eth.dst",
                COUNT(*)::BIGINT AS packet_count,
                SUM(p."frame.len")::BIGINT AS total_bytes
            FROM read_parquet('{escaped_path}') AS p
            WHERE
                p."eth.src" IN (SELECT mac FROM known_macs)
                OR
                p."eth.dst" IN (SELECT mac FROM known_macs)
            GROUP BY
                p."eth.src",
                p."eth.dst"
            """
        ).df()

    finally:
        con.close()

    elapsed = time.perf_counter() - start
    memory_mb = pairwise_df.memory_usage(deep=True).sum() / 1024**2

    print(
        f"[DONE] Produced {len(pairwise_df):,} MAC-pair rows "
        f"in {elapsed:.1f}s."
    )
    print(f"[INFO] Aggregate pandas memory: {memory_mb:.2f} MB")

    return pairwise_df


traffic_pairs_df = build_pairwise_traffic_summary(
    parquet_path=PARQUET_PATH,
    known_macs=device_dict_1.keys(),
    memory_limit="2GB",
    threads=4,
)


[START] Reading directly from: ../../../dataset/processed-data/input-data/dataset3/all_data.parquet
[INFO] parquet file size: 2.69 GB
[INFO] known devices: 50
[INFO] DuckDB memory limit: 2GB
[INFO] DuckDB threads: 4
[INFO] Raw packet rows will NOT be loaded into pandas.
[SCAN] Aggregating source/destination MAC pairs...
[DONE] Produced 846 MAC-pair rows in 4.9s.
[INFO] Aggregate pandas memory: 0.05 MB


In [11]:
print(
    f"[INFO] Aggregated table: {len(traffic_pairs_df):,} MAC pairs; "
    f"memory={traffic_pairs_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)
traffic_pairs_df.head(5)


[INFO] Aggregated table: 846 MAC pairs; memory=0.05 MB


,eth.src,eth.dst,packet_count,total_bytes
0,5c:aa:fd:6c:e0:d4,01:80:c2:00:00:00,2510694,153989232
1,00:17:88:21:f7:e4,d0:52:a8:63:47:9e,10349718,3488737176
2,00:01:c0:18:7f:9b,7c:64:56:60:71:74,1177053,1647908350
3,30:52:cb:a3:4f:5f,ff:ff:ff:ff:ff:ff,179157,11020096
4,7c:64:56:60:71:74,01:00:5e:00:00:07,40008,9655264


- A lot of traffic that are Ipv6 multicast neighbor discovery like 33:33:ff:xx:xx:xx. We'll classify if a mac is broadcast, unicast, or multicast
- If the mac are all FF -> broadcast
- Least significant bit of the first octect = 0 -> unicast
- Least significant bit of the first octect = 1 -> multicast

In [12]:
def classify_mac(mac):
    if pd.isna(mac):
        return "invalid"
    
    if mac in device_dict_1:
        return "unicast"
    
    mac = str(mac).lower().replace("-", ":")
    parts = mac.split(":")

    if len(parts) != 6:
        return "invalid"

    try:
        octets = [int(x, 16) for x in parts]
    except ValueError:
        return "invalid"

    if octets == [0xff] * 6:
        return "broadcast"

    if octets[0] & 0x01:
        return "multicast"

    return "unicast"

In [13]:
def compute_incoming_and_outgoing_data(
    pairwise_df: pd.DataFrame,
    device_dict_1: dict,
    log: bool = True,
) -> dict:
    """
    Build the same device_info structure as before, but from the small
    pre-aggregated MAC-pair table instead of rescanning raw packets.
    """
    device_info = {}
    known_macs = set(device_dict_1)

    # Index once so each lookup is against the small aggregated table.
    outgoing_groups = {
        mac: group
        for mac, group in pairwise_df.groupby("eth.src", sort=False)
        if mac in known_macs
    }
    incoming_groups = {
        mac: group
        for mac, group in pairwise_df.groupby("eth.dst", sort=False)
        if mac in known_macs
    }

    total_devices = len(device_dict_1)

    for i, source_mac in enumerate(device_dict_1, start=1):

        # ============================================================
        # Outgoing: source_mac -> peer
        # ============================================================
        source_rows = outgoing_groups.get(source_mac)

        if source_rows is None:
            outgoing_df = pd.DataFrame(
                columns=[
                    "peer_mac",
                    "outgoing_packet_count",
                    "outgoing_total_bytes",
                ]
            )
        else:
            outgoing_df = (
                source_rows[
                    ["eth.dst", "packet_count", "total_bytes"]
                ]
                .rename(
                    columns={
                        "eth.dst": "peer_mac",
                        "packet_count": "outgoing_packet_count",
                        "total_bytes": "outgoing_total_bytes",
                    }
                )
                .copy()
            )

        if not outgoing_df.empty:
            outgoing_df["peer_type"] = outgoing_df["peer_mac"].map(classify_mac)

            # Keep known unicast peers plus multicast/broadcast.
            outgoing_df = outgoing_df[
                ~(
                    outgoing_df["peer_type"].eq("unicast")
                    & ~outgoing_df["peer_mac"].isin(known_macs)
                )
            ].copy()

            outgoing_df.sort_values(
                "outgoing_packet_count",
                ascending=False,
                inplace=True,
                ignore_index=True,
            )
        else:
            outgoing_df["peer_type"] = pd.Series(dtype="object")

        outgoing_packet_total = outgoing_df["outgoing_packet_count"].sum()
        outgoing_byte_total = outgoing_df["outgoing_total_bytes"].sum()

        outgoing_df["outgoing_packet_count_pct"] = (
            outgoing_df["outgoing_packet_count"]
            .div(outgoing_packet_total if outgoing_packet_total else 1)
            .mul(100)
        )
        outgoing_df["outgoing_total_bytes_pct"] = (
            outgoing_df["outgoing_total_bytes"]
            .div(outgoing_byte_total if outgoing_byte_total else 1)
            .mul(100)
        )
        outgoing_df["direction"] = "outgoing"

        # ============================================================
        # Incoming: peer -> source_mac
        # ============================================================
        destination_rows = incoming_groups.get(source_mac)

        if destination_rows is None:
            incoming_df = pd.DataFrame(
                columns=[
                    "peer_mac",
                    "incoming_packet_count",
                    "incoming_total_bytes",
                ]
            )
        else:
            incoming_df = (
                destination_rows[
                    ["eth.src", "packet_count", "total_bytes"]
                ]
                .rename(
                    columns={
                        "eth.src": "peer_mac",
                        "packet_count": "incoming_packet_count",
                        "total_bytes": "incoming_total_bytes",
                    }
                )
                .copy()
            )

        if not incoming_df.empty:
            incoming_df["peer_type"] = incoming_df["peer_mac"].map(classify_mac)

            incoming_df = incoming_df[
                ~(
                    incoming_df["peer_type"].eq("unicast")
                    & ~incoming_df["peer_mac"].isin(known_macs)
                )
            ].copy()

            incoming_df.sort_values(
                "incoming_packet_count",
                ascending=False,
                inplace=True,
                ignore_index=True,
            )
        else:
            incoming_df["peer_type"] = pd.Series(dtype="object")

        incoming_packet_total = incoming_df["incoming_packet_count"].sum()
        incoming_byte_total = incoming_df["incoming_total_bytes"].sum()

        incoming_df["incoming_packet_count_pct"] = (
            incoming_df["incoming_packet_count"]
            .div(incoming_packet_total if incoming_packet_total else 1)
            .mul(100)
        )
        incoming_df["incoming_total_bytes_pct"] = (
            incoming_df["incoming_total_bytes"]
            .div(incoming_byte_total if incoming_byte_total else 1)
            .mul(100)
        )
        incoming_df["direction"] = "incoming"

        device_info[source_mac] = {
            "device_name": device_dict_1[source_mac]["device_name"],
            "outgoing_traffic_summary": outgoing_df,
            "incoming_traffic_summary": incoming_df,
        }

        if log:
            print(
                f"[DEVICE {i:>2}/{total_devices}] "
                f"{device_dict_1[source_mac]['device_name']} ({source_mac}) | "
                f"outgoing peers={len(outgoing_df):,}, "
                f"incoming peers={len(incoming_df):,}"
            )

    print(f"[DONE] Built traffic summaries for {len(device_info):,} devices.")
    return device_info


device_info = compute_incoming_and_outgoing_data(
    pairwise_df=traffic_pairs_df,
    device_dict_1=device_dict_1,
    log=True,
)


[DEVICE  1/50] Gateway (00:01:c0:18:7f:9b) | outgoing peers=53, incoming peers=49
[DEVICE  2/50] GoogleOnHub (f4:f2:6d:ce:9a:5d) | outgoing peers=6, incoming peers=3
[DEVICE  3/50] SamsungSmartThingsHub (d0:52:a8:63:47:9e) | outgoing peers=27, incoming peers=24
[DEVICE  4/50] PhilipsHUEHub (00:17:88:21:f7:e4) | outgoing peers=30, incoming peers=20
[DEVICE  5/50] InsteonHub (54:4a:16:f9:54:18) | outgoing peers=11, incoming peers=4
[DEVICE  6/50] Sonos (5c:aa:fd:6c:e0:d4) | outgoing peers=19, incoming peers=15
[DEVICE  7/50] SecurifiAlmond (e4:71:85:25:ce:ec) | outgoing peers=32, incoming peers=26
[DEVICE  8/50] NestCamera (18:b4:30:58:3d:6c) | outgoing peers=4, incoming peers=4
[DEVICE  9/50] BelkinWeMoMotionSensor (08:86:3b:6f:7a:15) | outgoing peers=9, incoming peers=7
[DEVICE 10/50] LIFXVirtualBulb (d0:73:d5:12:84:d1) | outgoing peers=8, incoming peers=4
[DEVICE 11/50] BelkinWeMoSwitch (08:86:3b:70:d7:39) | outgoing peers=9, incoming peers=7
[DEVICE 12/50] AmazonEchoGen1 (74:c2:46:1b

In [14]:
print(
    "Identifying potential routers based on unicast traffic "
    "and its corresponding traffic rates..."
)

potential_non_iot = set()

for mac in device_info:
    # if mac == GATEWAY_MAC:
    #     continue

    traffic_summary = device_info[mac]["outgoing_traffic_summary"]

    # Only consider known devices
    traffic_summary = traffic_summary[
        traffic_summary["peer_mac"].isin(device_dict_1.keys())
        & (traffic_summary["peer_mac"] != GATEWAY_MAC)
    ]

    # Only unicast destinations
    unicast_traffic = traffic_summary[
        traffic_summary["peer_type"] == "unicast"
    ]

    unicast_count = len(unicast_traffic)
    total_devices = len(device_dict_1) - 1 # Filter router

    if unicast_count / total_devices < 0.2:
        if device_dict_1[mac]["is_iot_device"] == 0:
            print(
                f"Missclassified {device_dict_1[mac]["device_name"]}: {mac}, "
                f"{unicast_count}/{total_devices} devices ({unicast_count / total_devices:.2%})\n"
                f"\tOutgoing traffic:\n"
            )
            display(device_info[mac]["outgoing_traffic_summary"].sort_values("peer_type"))
            print(f"\tIncoming traffic:\n")
            display(device_info[mac]["incoming_traffic_summary"].sort_values("peer_type"))
        continue
    potential_non_iot.add(mac)
    print(
        f"Potential non-IoT: "
        f"{device_info[mac]['device_name']} ({mac}) "
        f"- communicates with "
        f"{unicast_count}/{total_devices} devices "
        f"({unicast_count / total_devices:.2%})"
    )

correct_prediction = 0
all_device_but_gw = len(device_dict_1.keys())
for mac in device_dict_1:
    # if mac == GATEWAY_MAC:
    #     continue
    if mac in potential_non_iot and device_dict_1[mac]["is_iot_device"] == 0:
        correct_prediction += 1
    elif mac not in potential_non_iot and device_dict_1[mac]["is_iot_device"] == 1:
        correct_prediction += 1

print(f"Accuracy Rate: {correct_prediction / all_device_but_gw:.2%}")

Identifying potential routers based on unicast traffic and its corresponding traffic rates...
Potential non-IoT: Gateway (00:01:c0:18:7f:9b) - communicates with 49/49 devices (100.00%)
Potential non-IoT: SamsungSmartThingsHub (d0:52:a8:63:47:9e) - communicates with 24/49 devices (48.98%)
Potential non-IoT: PhilipsHUEHub (00:17:88:21:f7:e4) - communicates with 20/49 devices (40.82%)
Potential non-IoT: Sonos (5c:aa:fd:6c:e0:d4) - communicates with 15/49 devices (30.61%)
Potential non-IoT: SecurifiAlmond (e4:71:85:25:ce:ec) - communicates with 26/49 devices (53.06%)
Potential non-IoT: D-LinkDCS-5009LCamera (b0:c5:54:03:c7:09) - communicates with 11/49 devices (22.45%)
Potential non-IoT: MiCasaVerdeVeraLite (94:4a:0c:08:7e:72) - communicates with 37/49 devices (75.51%)
Potential non-IoT: Wink2Hub (00:21:cc:4d:59:35) - communicates with 35/49 devices (71.43%)
Missclassified AndroidTablet: a4:f1:e8:8d:b0:9e, 7/49 devices (14.29%)
	Outgoing traffic:



,peer_mac,outgoing_packet_count,outgoing_total_bytes,peer_type,outgoing_packet_count_pct,outgoing_total_bytes_pct,direction
7,ff:ff:ff:ff:ff:ff,420,43526,broadcast,0.038796,0.044072,outgoing
1,01:00:5e:00:00:fb,18207,4701096,multicast,1.681814,4.760036,outgoing
2,33:33:00:00:00:fb,17334,5040822,multicast,1.601173,5.104021,outgoing
4,33:33:00:00:00:02,2682,191124,multicast,0.247741,0.193520,outgoing
5,33:33:00:00:00:16,1068,98924,multicast,0.098653,0.100164,outgoing
6,01:00:5e:00:00:02,1059,64952,multicast,0.097822,0.065766,outgoing
11,33:33:ff:e6:69:45,27,2358,multicast,0.002494,0.002388,outgoing
12,01:00:5e:7f:ff:fa,15,2525,multicast,0.001386,0.002557,outgoing
14,33:33:ff:31:22:2a,6,524,multicast,0.000554,0.000531,outgoing
15,01:00:5e:00:00:fa,3,184,multicast,0.000277,0.000186,outgoing


	Incoming traffic:



,peer_mac,incoming_packet_count,incoming_total_bytes,peer_type,incoming_packet_count_pct,incoming_total_bytes_pct,direction
0,00:01:c0:18:7f:9b,2608738,3852377480,unicast,99.810765,99.972646,incoming
1,d4:90:9c:cc:62:42,3960,768720,unicast,0.151510,0.019949,incoming
2,e8:b2:ac:af:62:0f,414,86550,unicast,0.015840,0.002246,incoming
3,d0:03:4b:39:12:e3,364,168178,unicast,0.013927,0.004364,incoming
4,94:4a:0c:08:7e:72,68,4872,unicast,0.002602,0.000126,incoming
5,00:17:88:21:f7:e4,66,20952,unicast,0.002525,0.000544,incoming
6,00:21:cc:4d:59:35,56,3472,unicast,0.002143,0.000090,incoming
7,3c:f7:a4:f2:15:87,18,1312,unicast,0.000689,0.000034,incoming


Potential non-IoT: iPhone (3c:f7:a4:f2:15:87) - communicates with 15/49 devices (30.61%)
Potential non-IoT: iPad (e8:b2:ac:af:62:0f) - communicates with 47/49 devices (95.92%)
Accuracy Rate: 84.00%


In [15]:
print(f"Actual non-IoT devices: ")
for mac in device_dict_1:
    if device_dict_1[mac]["is_iot_device"] == 0 and mac != GATEWAY_MAC:
        print(f"\t{device_dict_1[mac]["device_name"]}: {mac}")

Actual non-IoT devices: 
	Gateway: 00:01:c0:18:7f:9b
	AndroidTablet: a4:f1:e8:8d:b0:9e
	iPhone: 3c:f7:a4:f2:15:87
	iPad: e8:b2:ac:af:62:0f


# Identify Router

In [ ]:
# unicast_summary = []

# for mac, info in device_info.items():

#     outgoing_df = info["outgoing_traffic_summary"]
#     incoming_df = info["incoming_traffic_summary"]

#     outgoing_unicast = outgoing_df[
#         outgoing_df["peer_type"] == "unicast"
#     ]

#     incoming_unicast = incoming_df[
#         incoming_df["peer_type"] == "unicast"
#     ]

#     outgoing_packets = outgoing_unicast["outgoing_packet_count"].sum()
#     outgoing_bytes = outgoing_unicast["outgoing_total_bytes"].sum()

#     incoming_packets = incoming_unicast["incoming_packet_count"].sum()
#     incoming_bytes = incoming_unicast["incoming_total_bytes"].sum()

#     outgoing_multicast = outgoing_df[
#         outgoing_df["peer_type"] == "multicast"
#     ]

#     outgoing_multicast_distinct_conversation = outgoing_multicast["peer_mac"].nunique()
#     outgoing_multicast_packets = outgoing_multicast["outgoing_packet_count"].sum()
#     outgoing_multicast_bytes = outgoing_multicast["outgoing_total_bytes"].sum()

#     unicast_summary.append({
#         "mac": mac,
#         "device_name": info["device_name"],
#         "outgoing_packet_count": outgoing_packets,
#         "outgoing_total_bytes": outgoing_bytes,
#         "incoming_packet_count": incoming_packets,
#         "incoming_total_bytes": incoming_bytes,
#         "total_packet_count": outgoing_packets + incoming_packets,
#         "total_bytes": outgoing_bytes + incoming_bytes,
#         "distinct_multicast_conversations": outgoing_multicast_distinct_conversation,
#         "outgoing_multicast_packet_count": outgoing_multicast_packets,
#         "outgoing_multicast_total_bytes": outgoing_multicast_bytes
#     })

# unicast_summary_df = pd.DataFrame(unicast_summary)

# display(unicast_summary_df.sort_values(by=["total_packet_count", "total_bytes"], ascending=False))

In [ ]:

# feature_cols = ["outgoing_packet_count","outgoing_total_bytes","incoming_packet_count","incoming_total_bytes", "total_packet_count", "total_bytes","distinct_multicast_conversations","outgoing_multicast_packet_count","outgoing_multicast_total_bytes"]
# # feature_cols = ["total_packet_count", "total_bytes","distinct_multicast_conversations","outgoing_multicast_total_bytes"]
# features = unicast_summary_df[
#     feature_cols
# ]

# # features = unicast_summary_df[
# #     ["total_packet_count"]
# # ]

# # X = np.log1p(features)
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(features)

# # kmeans = KMeans(
# #     n_clusters=2,
# #     random_state=42,
# #     n_init=20
# # )
# # unicast_summary_df["cluster"] = kmeans.fit_predict(X_scaled)

# # dbscan = DBSCAN(
# #     eps=1.5,
# #     min_samples=3
# # )

# # unicast_summary_df["cluster"] = dbscan.fit_predict(X_scaled)

# model = AgglomerativeClustering(
#     n_clusters=2
# )

# unicast_summary_df["cluster"] = (
#     model.fit_predict(X_scaled)
# )

# display(
#     unicast_summary_df[
#         [
#             "mac",
#             "device_name",
#             "cluster"
#         ] + feature_cols
#     ].sort_values("cluster")
# )

In [ ]:
# from itertools import combinations

# import pandas as pd
# import matplotlib.pyplot as plt


# def plotKMeans(
#     unicast_summary_df: pd.DataFrame,
#     feature_columns: list[str],
# ) -> None:

#     if len(feature_columns) < 2:
#         raise ValueError("feature_columns must contain at least 2 features.")

#     # Validate columns
#     required_columns = set(feature_columns) | {"cluster", "device_name"}
#     missing_columns = required_columns - set(unicast_summary_df.columns)

#     if missing_columns:
#         raise ValueError(
#             f"Missing columns in dataframe: {missing_columns}"
#         )

#     # Generate every pair of features
#     feature_pairs = combinations(feature_columns, 2)

#     for x_feature, y_feature in feature_pairs:

#         plt.figure(figsize=(10, 7))

#         # Plot each cluster
#         for cluster in sorted(unicast_summary_df["cluster"].unique()):

#             cluster_df = unicast_summary_df[
#                 unicast_summary_df["cluster"] == cluster
#             ]

#             plt.scatter(
#                 cluster_df[x_feature],
#                 cluster_df[y_feature],
#                 s=70,
#                 label=f"Cluster {cluster}"
#             )

#         # Label every device
#         for _, row in unicast_summary_df.iterrows():
#             plt.annotate(
#                 row["device_name"],
#                 (
#                     row[x_feature],
#                     row[y_feature]
#                 ),
#                 xytext=(5, 5),
#                 textcoords="offset points",
#                 fontsize=7
#             )

#         # Use log scale
#         plt.xscale("log")
#         plt.yscale("log")

#         plt.xlabel(x_feature)
#         plt.ylabel(y_feature)

#         plt.title(
#             f"Device Clustering: {x_feature} vs {y_feature}"
#         )

#         plt.legend()
#         plt.grid(alpha=0.3)
#         plt.tight_layout()
#         plt.savefig(
#             f"imgs/device_clustering_{x_feature}_vs_{y_feature}.png",
#             dpi=300,
#             bbox_inches="tight"
#         )
#         plt.show()
# plotKMeans(unicast_summary_df, feature_cols)

## Generate Mock Router

In [ ]:
# def create_mock_gateway_summary(
#     device_info,
#     unicast_summary_df,
#     device_dict,
#     gateway_mac,
#     n_gateways=3,
#     random_state=42,
# ):
#     if n_gateways < 1:
#         raise ValueError("n_gateways must be >= 1")

#     rng = np.random.default_rng(random_state)

#     # ============================================================
#     # Create synthetic gateway MACs
#     # ============================================================

#     mock_gateways = [
#         f"02:00:00:00:00:{i + 1:02x}"
#         for i in range(n_gateways)
#     ]

#     main_gateway = rng.choice(mock_gateways)

#     # ============================================================
#     # Randomly assign devices to gateways
#     # ============================================================

#     endpoint_devices = [
#         mac
#         for mac in device_dict
#         if mac != gateway_mac
#     ]

#     shuffled = endpoint_devices.copy()
#     rng.shuffle(shuffled)

#     device_to_gateway = {
#         mac: mock_gateways[i % n_gateways]
#         for i, mac in enumerate(shuffled)
#     }

#     # ============================================================
#     # Get original gateway <-> device traffic
#     # ============================================================

#     gateway_info = device_info[gateway_mac]

#     outgoing = gateway_info[
#         "outgoing_traffic_summary"
#     ].copy()

#     incoming = gateway_info[
#         "incoming_traffic_summary"
#     ].copy()

#     # Keep known unicast devices only
#     outgoing = outgoing[
#         (outgoing["peer_type"] == "unicast")
#         & outgoing["peer_mac"].isin(endpoint_devices)
#     ].copy()

#     incoming = incoming[
#         (incoming["peer_type"] == "unicast")
#         & incoming["peer_mac"].isin(endpoint_devices)
#     ].copy()

#     # ============================================================
#     # Make one row per peer
#     # ============================================================

#     outgoing = outgoing[
#         [
#             "peer_mac",
#             "outgoing_packet_count",
#             "outgoing_total_bytes",
#         ]
#     ]

#     incoming = incoming[
#         [
#             "peer_mac",
#             "incoming_packet_count",
#             "incoming_total_bytes",
#         ]
#     ]

#     peer_traffic = pd.merge(
#         outgoing,
#         incoming,
#         on="peer_mac",
#         how="outer",
#     ).fillna(0)

#     peer_traffic["assigned_gateway"] = (
#         peer_traffic["peer_mac"]
#         .map(device_to_gateway)
#     )

#     # ============================================================
#     # Aggregate traffic assigned to each gateway
#     # ============================================================

#     gateway_load = (
#         peer_traffic
#         .groupby("assigned_gateway", as_index=False)
#         .agg(
#             original_outgoing_packets=(
#                 "outgoing_packet_count",
#                 "sum"
#             ),
#             original_outgoing_bytes=(
#                 "outgoing_total_bytes",
#                 "sum"
#             ),
#             original_incoming_packets=(
#                 "incoming_packet_count",
#                 "sum"
#             ),
#             original_incoming_bytes=(
#                 "incoming_total_bytes",
#                 "sum"
#             ),
#         )
#         .set_index("assigned_gateway")
#         .reindex(mock_gateways, fill_value=0)
#     )

#     # ============================================================
#     # Construct synthetic gateway totals
#     # ============================================================

#     gateway_rows = []

#     # Original gateway directional totals
#     original_total_out_packets = (
#         peer_traffic["outgoing_packet_count"].sum()
#     )

#     original_total_out_bytes = (
#         peer_traffic["outgoing_total_bytes"].sum()
#     )

#     original_total_in_packets = (
#         peer_traffic["incoming_packet_count"].sum()
#     )

#     original_total_in_bytes = (
#         peer_traffic["incoming_total_bytes"].sum()
#     )

#     for i, gw in enumerate(mock_gateways):

#         load = gateway_load.loc[gw]

#         if gw == main_gateway:

#             # Main gateway sees every original gateway packet once
#             outgoing_packets = original_total_out_packets
#             outgoing_bytes = original_total_out_bytes

#             incoming_packets = original_total_in_packets
#             incoming_bytes = original_total_in_bytes

#             name = f"Mock Gateway {i + 1} (Main)"

#         else:

#             # Downstream gateway participates on BOTH links:
#             #
#             # device <-> downstream GW
#             # downstream GW <-> main GW
#             #
#             # Every original packet therefore contributes
#             # one incoming + one outgoing packet.

#             traffic_packets = (
#                 load["original_outgoing_packets"]
#                 + load["original_incoming_packets"]
#             )

#             traffic_bytes = (
#                 load["original_outgoing_bytes"]
#                 + load["original_incoming_bytes"]
#             )

#             outgoing_packets = traffic_packets
#             incoming_packets = traffic_packets

#             outgoing_bytes = traffic_bytes
#             incoming_bytes = traffic_bytes

#             name = f"Mock Gateway {i + 1}"

#         gateway_rows.append({
#             "mac": gw,
#             "device_name": name,

#             "outgoing_packet_count": outgoing_packets,
#             "outgoing_total_bytes": outgoing_bytes,

#             "incoming_packet_count": incoming_packets,
#             "incoming_total_bytes": incoming_bytes,

#             "total_packet_count": (
#                 outgoing_packets
#                 + incoming_packets
#             ),

#             "total_bytes": (
#                 outgoing_bytes
#                 + incoming_bytes
#             ),

#             "is_mock_gateway": True,
#             "is_main_gateway": gw == main_gateway,
#         })

#     gateway_summary_df = pd.DataFrame(gateway_rows)

#     # ============================================================
#     # Keep original endpoint totals
#     # ============================================================

#     device_summary = unicast_summary_df[
#         unicast_summary_df["mac"] != gateway_mac
#     ].copy()

#     device_summary["is_mock_gateway"] = False
#     device_summary["is_main_gateway"] = False

#     # ============================================================
#     # Combine endpoints + synthetic gateways
#     # ============================================================

#     result_df = pd.concat(
#         [
#             device_summary,
#             gateway_summary_df,
#         ],
#         ignore_index=True,
#         sort=False,
#     )

#     # ============================================================
#     # Assignment table
#     # ============================================================

#     assignment_df = pd.DataFrame([
#         {
#             "device_mac": mac,
#             "device_name": device_dict[mac]["device_name"],
#             "assigned_gateway": gw,
#             "assigned_to_main": gw == main_gateway,
#         }
#         for mac, gw in device_to_gateway.items()
#     ])

#     return (
#         result_df,
#         gateway_summary_df,
#         assignment_df,
#         main_gateway,
#         peer_traffic,
#     )

In [ ]:
# def create_mock_gateway_summary(
#     device_info,
#     unicast_summary_df,
#     device_dict,
#     gateway_mac,
#     n_gateways=3,
#     random_state=42,
# ):
#     if n_gateways < 1:
#         raise ValueError("n_gateways must be >= 1")

#     rng = np.random.default_rng(random_state)

#     # ============================================================
#     # Create synthetic gateway MACs
#     # ============================================================

#     mock_gateways = [
#         f"02:00:00:00:00:{i + 1:02x}"
#         for i in range(n_gateways)
#     ]

#     main_gateway = rng.choice(mock_gateways)

#     # ============================================================
#     # Randomly assign devices to gateways
#     # ============================================================

#     endpoint_devices = [
#         mac
#         for mac in device_dict
#         if mac != gateway_mac
#     ]

#     shuffled = endpoint_devices.copy()
#     rng.shuffle(shuffled)

#     device_to_gateway = {
#         mac: mock_gateways[i % n_gateways]
#         for i, mac in enumerate(shuffled)
#     }

#     # ============================================================
#     # Get original gateway traffic
#     # ============================================================

#     gateway_info = device_info[gateway_mac]

#     outgoing = gateway_info[
#         "outgoing_traffic_summary"
#     ].copy()

#     incoming = gateway_info[
#         "incoming_traffic_summary"
#     ].copy()

#     # ============================================================
#     # Extract original gateway multicast traffic
#     #
#     # Assumption:
#     # Every mock gateway participates in the same multicast
#     # conversations and therefore gets identical multicast features.
#     # ============================================================

#     outgoing_multicast = outgoing[
#         outgoing["peer_type"] == "multicast"
#     ].copy()

#     original_multicast_conversations = (
#         outgoing_multicast["peer_mac"].nunique()
#     )

#     original_multicast_packets = (
#         outgoing_multicast["outgoing_packet_count"].sum()
#     )

#     original_multicast_bytes = (
#         outgoing_multicast["outgoing_total_bytes"].sum()
#     )

#     # ============================================================
#     # Keep known unicast devices only
#     # ============================================================

#     outgoing_unicast = outgoing[
#         (outgoing["peer_type"] == "unicast")
#         & outgoing["peer_mac"].isin(endpoint_devices)
#     ].copy()

#     incoming_unicast = incoming[
#         (incoming["peer_type"] == "unicast")
#         & incoming["peer_mac"].isin(endpoint_devices)
#     ].copy()

#     # ============================================================
#     # Keep only columns needed for peer-level traffic
#     # ============================================================

#     outgoing_unicast = outgoing_unicast[
#         [
#             "peer_mac",
#             "outgoing_packet_count",
#             "outgoing_total_bytes",
#         ]
#     ]

#     incoming_unicast = incoming_unicast[
#         [
#             "peer_mac",
#             "incoming_packet_count",
#             "incoming_total_bytes",
#         ]
#     ]

#     # ============================================================
#     # Merge outgoing and incoming traffic
#     #
#     # One row per peer device
#     # ============================================================

#     peer_traffic = pd.merge(
#         outgoing_unicast,
#         incoming_unicast,
#         on="peer_mac",
#         how="outer",
#     ).fillna(0)

#     # Assign each peer device to a mock gateway
#     peer_traffic["assigned_gateway"] = (
#         peer_traffic["peer_mac"]
#         .map(device_to_gateway)
#     )

#     # ============================================================
#     # Aggregate original gateway <-> device traffic
#     # for each mock gateway
#     # ============================================================

#     gateway_load = (
#         peer_traffic
#         .groupby(
#             "assigned_gateway",
#             as_index=False,
#         )
#         .agg(
#             original_outgoing_packets=(
#                 "outgoing_packet_count",
#                 "sum",
#             ),
#             original_outgoing_bytes=(
#                 "outgoing_total_bytes",
#                 "sum",
#             ),
#             original_incoming_packets=(
#                 "incoming_packet_count",
#                 "sum",
#             ),
#             original_incoming_bytes=(
#                 "incoming_total_bytes",
#                 "sum",
#             ),
#         )
#         .set_index("assigned_gateway")
#         .reindex(
#             mock_gateways,
#             fill_value=0,
#         )
#     )

#     # ============================================================
#     # Original gateway directional UNICAST totals
#     # ============================================================

#     original_total_out_packets = (
#         peer_traffic["outgoing_packet_count"].sum()
#     )

#     original_total_out_bytes = (
#         peer_traffic["outgoing_total_bytes"].sum()
#     )

#     original_total_in_packets = (
#         peer_traffic["incoming_packet_count"].sum()
#     )

#     original_total_in_bytes = (
#         peer_traffic["incoming_total_bytes"].sum()
#     )

#     # ============================================================
#     # Construct synthetic gateway summaries
#     # ============================================================

#     gateway_rows = []

#     for i, gw in enumerate(mock_gateways):

#         load = gateway_load.loc[gw]

#         # ========================================================
#         # Main gateway
#         # ========================================================

#         if gw == main_gateway:

#             # Main gateway sees all original gateway traffic
#             # because downstream gateway traffic eventually
#             # reaches the main gateway.
#             outgoing_packets = (
#                 original_total_out_packets
#             )

#             outgoing_bytes = (
#                 original_total_out_bytes
#             )

#             incoming_packets = (
#                 original_total_in_packets
#             )

#             incoming_bytes = (
#                 original_total_in_bytes
#             )

#             name = f"Mock Gateway {i + 1} (Main)"

#         # ========================================================
#         # Downstream gateway
#         # ========================================================

#         else:

#             # A downstream gateway participates in two links:
#             #
#             # device <-> downstream gateway
#             #
#             # downstream gateway <-> main gateway
#             #
#             # Therefore, every original packet associated with
#             # this downstream gateway contributes one incoming
#             # and one outgoing packet from the downstream
#             # gateway's perspective.

#             traffic_packets = (
#                 load["original_outgoing_packets"]
#                 + load["original_incoming_packets"]
#             )

#             traffic_bytes = (
#                 load["original_outgoing_bytes"]
#                 + load["original_incoming_bytes"]
#             )

#             outgoing_packets = traffic_packets
#             incoming_packets = traffic_packets

#             outgoing_bytes = traffic_bytes
#             incoming_bytes = traffic_bytes

#             name = f"Mock Gateway {i + 1}"

#         # ========================================================
#         # Multicast features
#         #
#         # Replicate original gateway multicast behavior across
#         # ALL mock gateways.
#         # ========================================================

#         distinct_multicast_conversations = (
#             original_multicast_conversations
#         )

#         outgoing_multicast_packet_count = (
#             original_multicast_packets
#         )

#         outgoing_multicast_total_bytes = (
#             original_multicast_bytes
#         )

#         # ========================================================
#         # Create gateway summary row
#         # ========================================================

#         gateway_rows.append({
#             "mac": gw,

#             "device_name": name,

#             # ----------------------------
#             # Unicast features
#             # ----------------------------

#             "outgoing_packet_count": (
#                 outgoing_packets
#             ),

#             "outgoing_total_bytes": (
#                 outgoing_bytes
#             ),

#             "incoming_packet_count": (
#                 incoming_packets
#             ),

#             "incoming_total_bytes": (
#                 incoming_bytes
#             ),

#             "total_packet_count": (
#                 outgoing_packets
#                 + incoming_packets
#             ),

#             "total_bytes": (
#                 outgoing_bytes
#                 + incoming_bytes
#             ),

#             # ----------------------------
#             # Multicast features
#             # ----------------------------

#             "distinct_multicast_conversations": (
#                 distinct_multicast_conversations
#             ),

#             "outgoing_multicast_packet_count": (
#                 outgoing_multicast_packet_count
#             ),

#             "outgoing_multicast_total_bytes": (
#                 outgoing_multicast_total_bytes
#             ),

#             # ----------------------------
#             # Metadata
#             # ----------------------------

#             "is_mock_gateway": True,

#             "is_main_gateway": (
#                 gw == main_gateway
#             ),
#         })

#     # ============================================================
#     # Gateway summary dataframe
#     # ============================================================

#     gateway_summary_df = pd.DataFrame(
#         gateway_rows
#     )

#     # ============================================================
#     # Keep original endpoint device summaries
#     #
#     # Remove the real gateway because it is being replaced by
#     # the mock gateways.
#     # ============================================================

#     device_summary = unicast_summary_df[
#         unicast_summary_df["mac"] != gateway_mac
#     ].copy()

#     device_summary[
#         "is_mock_gateway"
#     ] = False

#     device_summary[
#         "is_main_gateway"
#     ] = False

#     # ============================================================
#     # Combine endpoint devices + synthetic gateways
#     # ============================================================

#     result_df = pd.concat(
#         [
#             device_summary,
#             gateway_summary_df,
#         ],
#         ignore_index=True,
#         sort=False,
#     )

#     # ============================================================
#     # Create device -> gateway assignment table
#     # ============================================================

#     assignment_df = pd.DataFrame([
#         {
#             "device_mac": mac,

#             "device_name": (
#                 device_dict[mac]["device_name"]
#             ),

#             "assigned_gateway": gw,

#             "assigned_to_main": (
#                 gw == main_gateway
#             ),
#         }
#         for mac, gw in device_to_gateway.items()
#     ])

#     # ============================================================
#     # Return results
#     # ============================================================

#     return (
#         result_df,
#         gateway_summary_df,
#         assignment_df,
#         main_gateway,
#         peer_traffic,
#     )

In [ ]:
NUMBER_MOCK_GATEWAY = 3

In [ ]:
(
    mock_summary_df,
    gateway_summary_df,
    assignment_df,
    main_gateway,
    peer_traffic,
) = create_mock_gateway_summary(
    device_info=device_info,
    unicast_summary_df=unicast_summary_df,
    device_dict=device_dict_1,
    gateway_mac=GATEWAY_MAC,
    n_gateways=NUMBER_MOCK_GATEWAY,
    random_state=42,
)

In [ ]:
mock_summary_df

In [ ]:
# mock_summary_df["multicast_outgoing_packet_count_per_device"] = (
#     mock_summary_df["outgoing_multicast_packet_count"]
#     .div(mock_summary_df["distinct_multicast_conversations"])
#     .fillna(0)
# )

# mock_summary_df["multicast_outgoing_bytes_per_device"] = (
#     mock_summary_df["outgoing_multicast_total_bytes"]
#     .div(mock_summary_df["distinct_multicast_conversations"])
#     .fillna(0)
# )

# mock_summary_df["multicast_outgoing_bytes_per_packet"] = (
#     mock_summary_df["outgoing_multicast_total_bytes"]
#     .div(mock_summary_df["outgoing_multicast_packet_count"])
#     .fillna(0)
# )
# EXTRA_COLS = ["multicast_outgoing_packet_count_per_device","multicast_outgoing_bytes_per_device","multicast_outgoing_bytes_per_packet"]

In [ ]:
# def feature_selection(all_features: List[str], mock_gate_way_num: int, df: pd.DataFrame) -> List[List[str]]:
#     feature_combinations = [
#         list(combo)
#         for r in range(1, len(all_features) + 1)
#         for combo in combinations(all_features, r)
#     ]
#     valid_combos = []
#     for combo in feature_combinations:
#         features_with_mac = df[
#             feature_cols + ["device_name", "mac"]
#         ]
#         features = df[
#             feature_cols
#         ]
#         scaler = StandardScaler()
#         X_scaled = scaler.fit_transform(features)

#         # kmeans = KMeans(
#         #     n_clusters=2,
#         #     random_state=42,
#         #     n_init=20
#         # )
        
#         # features_with_mac["cluster"] = kmeans.fit_predict(X_scaled)

#         model = AgglomerativeClustering(
#             n_clusters=2
#         )

#         features_with_mac["cluster"] = (
#             model.fit_predict(X_scaled)
#         )

#         # query for all routers
#         router_summary_df = features_with_mac[features_with_mac["device_name"].str.startswith("Mock Gateway", na=False)]
#         router_cluster_id = router_summary_df["cluster"].value_counts().idxmax()

#         total_correctly_classified_router = len(router_summary_df[router_summary_df["cluster"] == router_cluster_id])
#         all_device_classified_as_router = len(features_with_mac[features_with_mac["cluster"] == router_cluster_id])

#         if total_correctly_classified_router == all_device_classified_as_router and total_correctly_classified_router == mock_
#         _num:
#             print(f"{combo}:\n\ttotal_correctly_classified_router: {total_correctly_classified_router}\t\tall_device_classified_as_router: {all_device_classified_as_router}")
#             valid_combos.append(combo)

#     return valid_combos

# valid_feature_combos = feature_selection(feature_cols + EXTRA_COLS, NUMBER_MOCK_GATEWAY, mock_summary_df)

# valid_feature_combos

# # display(
# #     mock_summary_df[
# #         [
# #             "mac",
# #             "device_name",
# #             "cluster"
# #         ] + feature_cols
# #     ].sort_values("cluster")
# # )
# # plotKMeans(mock_summary_df, feature_cols)

- I've tried different configuration and laptop keeps showing up which means we can not only rely on the volume (packet count, packet bytes). There can be something else we should add to differentiate macbook from the gateway.
- Another feature we can add is the number of devices communicating with this device. Usually the router will communicate with more devices. So that might help in the clustering?

# Note on September 15th, 2026
- Add more features/dimensions for the KMeans
- Try to cluster IoT vs non-IoT devices
- Try Kaggle dataset

# Note on September 17th, 2026
-  How many unique devices? Threshold on packet count, packet bytes, packet rate.

# Note
- Trying every possible combination but can not separate it with KMeans
- KMeans still misclassified when we run for 2 mock devices
- Agglomerative is better at 2 devices but can not classify 3 devices
- Normal features doesn't seem to work because in the current dataset, the non-IoT devices are not really active, thus overall traffic volume would kind of be the same as the IoT device. However, one thing we can look at is the packet rate? 

# Now trying IoT and non-IoT

In [ ]:
# from sklearn.mixture import GaussianMixture
# from sklearn.cluster import SpectralClustering
# feature_cols = ["outgoing_packet_count","outgoing_total_bytes","incoming_packet_count","incoming_total_bytes", "total_packet_count", "total_bytes","distinct_multicast_conversations","outgoing_multicast_packet_count","outgoing_multicast_total_bytes"]
# # feature_cols = ["total_packet_count", "total_bytes","distinct_multicast_conversations","outgoing_multicast_total_bytes"]
# unicast_summary_no_gw = unicast_summary_df[unicast_summary_df["mac"] != GATEWAY_MAC]

In [ ]:
# unicast_summary_no_gw["multicast_outgoing_packet_count_per_device"] = (
#     unicast_summary_no_gw["outgoing_multicast_packet_count"]
#     .div(unicast_summary_no_gw["distinct_multicast_conversations"])
#     .fillna(0)
# )

# unicast_summary_no_gw["multicast_outgoing_bytes_per_device"] = (
#     unicast_summary_no_gw["outgoing_multicast_total_bytes"]
#     .div(unicast_summary_no_gw["distinct_multicast_conversations"])
#     .fillna(0)
# )

# unicast_summary_no_gw["multicast_outgoing_bytes_per_packet"] = (
#     unicast_summary_no_gw["outgoing_multicast_total_bytes"]
#     .div(unicast_summary_no_gw["outgoing_multicast_packet_count"])
#     .fillna(0)
# )
# EXTRA_COLS = ["multicast_outgoing_packet_count_per_device","multicast_outgoing_bytes_per_device","multicast_outgoing_bytes_per_packet"]

In [ ]:
# unicast_summary_no_gw

In [ ]:
# features = unicast_summary_no_gw[EXTRA_COLS + feature_cols]

In [ ]:


# # features = unicast_summary_df[
# #     ["total_packet_count"]
# # ]

# # X = np.log1p(features)
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(features)

# # kmeans = KMeans(
# #     n_clusters=2,
# #     random_state=42,
# #     n_init=20
# # )
# # unicast_summary_no_gw["cluster"] = kmeans.fit_predict(X_scaled)

# # dbscan = DBSCAN(
# #     eps=1.5,
# #     min_samples=3
# # )

# # unicast_summary_no_gw["cluster"] = dbscan.fit_predict(X_scaled)

# model = AgglomerativeClustering(
#     n_clusters=2
# )

# unicast_summary_no_gw["cluster"] = (
#     model.fit_predict(X_scaled)
# )

# # model = GaussianMixture(
# #     n_components=2,
# #     random_state=42
# # )

# # unicast_summary_no_gw["cluster"] = (
# #     model.fit_predict(X_scaled)
# # )

# # model = SpectralClustering(
# #     n_clusters=2,
# #     affinity="nearest_neighbors",
# #     random_state=42
# # )

# # unicast_summary_no_gw["cluster"] = (
# #     model.fit_predict(X_scaled)
# # )

# display(
#     unicast_summary_no_gw[
#         [
#             "mac",
#             "device_name",
#             "cluster"
#         ] + feature_cols
#     ].sort_values("cluster")
# )

In [ ]:
# plotKMeans(unicast_summary_no_gw, feature_cols)